# MedVision Thesis — Part 1: Data Pipeline

**Downloads PhysioNet metadata + 6,000 images, builds cohort, creates patient-level split.**

## Setup
1. Set Kaggle Secrets: PHYSIONET_USER, PHYSIONET_PASS
2. Enable Internet + GPU T4 x2
3. Run All (~2-3 hours)
4. Save Version > Quick Save

## Dataset Budget (6,000 images)
| Class | Cap |
|-------|-----|
| Normal | 2,500 |
| Pneumonia | 2,500 |
| Pneumothorax | 1,000 |
| **Total** | **6,000** |

Split: 70/15/15 by patient. Test ~900 (PNX ~150).


In [1]:
!pip install -q transformers==4.44.2 datasets torchvision scikit-learn matplotlib seaborn nltk tqdm statsmodels nbformat requests torchxrayvision

import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['MALLOC_TRIM_THRESHOLD_'] = '0'

import importlib
for pkg in ['transformers', 'datasets', 'statsmodels', 'torchxrayvision']:
    try:
        importlib.import_module(pkg)
        print(f"  [OK] {pkg}")
    except ImportError:
        raise ImportError(f"Package '{pkg}' failed to install.")
print("All dependencies installed.")

import torch
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 85.8 MB/s eta 0:00:00
  [OK] transformers
  [OK] datasets
  [OK] statsmodels
  [OK] torchxrayvision
All dependencies installed.
  GPU: Tesla T4


In [2]:
import os, sys, json, random, re, copy, time, warnings, math, io, shutil, gc, hashlib, zipfile, pickle, subprocess
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler
import torchvision
from torchvision import transforms
from transformers import AutoModel, AutoTokenizer
from sklearn.model_selection import GroupShuffleSplit, StratifiedKFold, cross_val_predict
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report, brier_score_loss)
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy import stats as scipy_stats
from scipy.stats import wilcoxon, ttest_rel
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.proportion import proportion_confint
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import requests
import torchxrayvision as xrv

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)

def mem_stats(label=""):
    ram_mb = 0
    try:
        import resource
        ram_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
    except: pass
    gpu_mb = torch.cuda.memory_allocated() / 1e6 if torch.cuda.is_available() else 0
    if label: print(f"  [mem {label}] RAM={ram_mb:.0f}MB, GPU={gpu_mb:.0f}MB")

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"torchxrayvision: {xrv.__version__}")
print("Imports OK.")


PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4
torchxrayvision: 1.5.2
Imports OK.


In [3]:
TB_DECISION = 'drop'
PHASE0_FREEZE_LMH = True
PHASE0_ACCEPT_NEGATIVE = True

if TB_DECISION == 'unset':
    raise RuntimeError("Set TB_DECISION to 'drop' or 'validate'.")
if TB_DECISION == 'keep':
    raise RuntimeError("TB_DECISION='keep' is forbidden.")

class Config:
    SEED = 42
    SEED_LIST = [42, 123, 456, 789, 1010, 1111, 1212, 1313]
    ABLATION_SEED_LIST = [42, 123, 456]
    BASELINE_EPOCHS = 10
    LMH_EPOCHS = 10
    GACR_B_EPOCHS = 10
    NOISE_CONSISTENCY_EPOCHS = 10
    ADAPTIVE_CADQ_EPOCHS = 10
    ABLATION_EPOCHS = 10
    LMH_G_MAX = 5.0
    LMH_LAMBDA_ENTROPY = 0.05
    LMH_WARM_START = True
    LMH_ANNEAL_EPOCHS = 3
    LMH_BIAS_EPSILON = 0.1
    LMH_BIAS_TEMP = 2.0
    LMH_CALIBRATE_BIAS = True
    LMH_G_INIT_BIAS = -2.0
    LMH_ASSERT_LOSS_POSITIVE = True
    LR_OOF_FOLDS = 5
    DATA_SOURCE = 'physionet'
    PHYSIONET_RECORD_LIST = '/kaggle/working/physionet_metadata/cxr-record-list.csv.gz'
    PHYSIONET_STUDY_LIST = '/kaggle/working/physionet_metadata/cxr-study-list.csv.gz'
    PHYSIONET_REPORTS_ZIP = '/kaggle/working/physionet_metadata/mimic-cxr-reports.zip'
    PHYSIONET_JPG_METADATA = '/kaggle/working/physionet_metadata/mimic-cxr-2.0.0-metadata.csv.gz'
    PHYSIONET_JPG_SPLIT = '/kaggle/working/physionet_metadata/mimic-cxr-2.0.0-split.csv.gz'
    PHYSIONET_JPG_CHEXPERT = '/kaggle/working/physionet_metadata/mimic-cxr-2.0.0-chexpert.csv.gz'
    PHYSIONET_IMAGE_BASE = 'https://physionet.org/files/mimic-cxr-jpg/2.1.0/'
    PHYSIONET_IMAGE_DIR = '/kaggle/working/images'
    FRONTAL_ONLY = True
    ALLOW_HF_FALLBACK = False
    NORMAL_CAP = 2500
    PNEUMONIA_CAP = 2500
    PNEUMOTHORAX_CAP = 1000
    TEXT_MAX_LEN = 256
    IMG_SIZE = 224
    PROJECTION_DIM = 512
    NUM_HEADS = 8
    NUM_CLASSES = 3
    LABEL_NAMES = ['Normal', 'Pneumonia', 'Pneumothorax']
    TEXT_ENCODER = 'emilyalsentzer/Bio_ClinicalBERT'
    TEXT_FROZEN_LAYERS = 4
    IMAGE_FROZEN_STEM_ONLY = True
    DROPOUT_RATE = 0.25
    BATCH_SIZE = 8
    GRAD_ACCUM_STEPS = 4
    LR_IMG_BACKBONE = 1e-5
    LR_IMG_HEAD = 5e-5
    LR_TEXT = 1e-5
    LR_FUSION = 1e-4
    LR_GATE = 5e-5
    LR_G_HEAD = 1e-4
    WEIGHT_DECAY = 1e-4
    WARMUP_RATIO = 0.1
    LABEL_SMOOTHING = 0.05
    FOCAL_GAMMA = 1.5
    USE_CLASS_WEIGHTS = True
    GATE_GRAD_CLIP = 10.0
    TB_PER_BATCH = 2
    TB_OVERSAMPLE = 5
    PN_OVERSAMPLE = 3
    CLASS_WEIGHT_BETA = 0.99
    CADQ_MIN_IMAGE_GATE = 0.20
    CADQ_MAX_IMAGE_GATE = 0.80
    GATE_INIT_LOGITS = [-0.5, 0.0, 0.5]
    STOCHASTIC_TEXT_DROP = 0.10
    NOISE_CONSISTENCY_LAMBDA = 0.3
    NOISE_CONSISTENCY_WARMUP = 2
    SAVE_DIR = '/kaggle/working/checkpoints'
    RESULTS_DIR = '/kaggle/working/results'
    IMAGE_CACHE_DIR = '/kaggle/working/img_cache'

cfg = Config()
for d in [cfg.SAVE_DIR, cfg.RESULTS_DIR, cfg.IMAGE_CACHE_DIR, cfg.PHYSIONET_IMAGE_DIR]:
    if os.path.exists(d): shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(cfg.SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"  Caps: N={cfg.NORMAL_CAP}, PN={cfg.PNEUMONIA_CAP}, PNX={cfg.PNEUMOTHORAX_CAP}")
print(f"  Total budget: {cfg.NORMAL_CAP + cfg.PNEUMONIA_CAP + cfg.PNEUMOTHORAX_CAP}")
print(f"  Device: {device}")


  Caps: N=2500, PN=2500, PNX=1000
  Total budget: 6000
  Device: cuda


In [4]:
def split_report_sections(report_text):
    if not report_text or not isinstance(report_text, str): return '', ''
    text = report_text.strip()
    findings_match = re.search(r'(?i)\bfindings?\s*[:\-]\s*(.*?)(?=\s*\bimpression\s*[:\-]\b|$)', text, re.DOTALL)
    impression_match = re.search(r'(?i)\bimpression\s*[:\-]\s*(.*?)$', text, re.DOTALL)
    if findings_match and impression_match:
        f, i = findings_match.group(1).strip(), impression_match.group(1).strip()
        if f and i: return f, i
    if '  ' in text:
        parts = text.split('  ', 1)
        f, i = parts[0].strip(), parts[1].strip() if len(parts) > 1 else ''
        if len(f) > 20 and len(i) > 5: return f, i
    if '\n\n' in text:
        parts = text.split('\n\n', 1)
        f = parts[0].strip()
        i = parts[1].strip() if len(parts) > 1 else ''
        if len(f) > 20 and len(i) > 5: return f, i
    sentences = re.split(r'(?<=[.!?])\s+', text)
    if len(sentences) >= 3:
        i = sentences[-1].strip()
        f = ' '.join(sentences[:-1]).strip()
        if len(f) > 20 and len(i) > 5: return f, i
    return text, ''
print("Report splitter defined.")

# ============================================================
# PhysioNet download + cohort + split + image download + save
# ============================================================
def get_physionet_credentials():
    user = os.environ.get('PHYSIONET_USER', '')
    pwd = os.environ.get('PHYSIONET_PASS', '')
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        user = secrets.get_secret('PHYSIONET_USER') or user
        pwd = secrets.get_secret('PHYSIONET_PASS') or pwd
        if user and pwd: print("  [OK] PhysioNet credentials loaded")
    except: pass
    if not user or not pwd:
        from getpass import getpass
        user = input("PhysioNet username: ")
        pwd = getpass("PhysioNet password: ")
    return user, pwd

def wget_download(url, out_path, user, pwd, max_retries=5):
    for attempt in range(max_retries):
        cmd = ['wget', '-c', '--user=' + user, '--password=' + pwd, '--progress=dot:giga', '-O', out_path, url]
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
            if result.returncode == 0 and os.path.exists(out_path) and os.path.getsize(out_path) > 1000:
                print(f"    [OK] {os.path.basename(out_path)} ({os.path.getsize(out_path)/1e6:.1f} MB)")
                return True
        except subprocess.TimeoutExpired:
            print(f"    [WARN] Attempt {attempt+1} timed out. Re-run resumes.")
        except Exception as e:
            print(f"    [WARN] Attempt {attempt+1}: {e}")
    return False

def ensure_physionet_metadata(cfg):
    needed = [
        ('cxr-record-list.csv.gz', cfg.PHYSIONET_RECORD_LIST, 'https://physionet.org/files/mimic-cxr/2.0.0/cxr-record-list.csv.gz'),
        ('cxr-study-list.csv.gz', cfg.PHYSIONET_STUDY_LIST, 'https://physionet.org/files/mimic-cxr/2.0.0/cxr-study-list.csv.gz'),
        ('mimic-cxr-reports.zip', cfg.PHYSIONET_REPORTS_ZIP, 'https://physionet.org/files/mimic-cxr/2.0.0/mimic-cxr-reports.zip'),
        ('mimic-cxr-2.0.0-metadata.csv.gz', cfg.PHYSIONET_JPG_METADATA, 'https://physionet.org/files/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-metadata.csv.gz'),
        ('mimic-cxr-2.0.0-split.csv.gz', cfg.PHYSIONET_JPG_SPLIT, 'https://physionet.org/files/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-split.csv.gz'),
        ('mimic-cxr-2.0.0-chexpert.csv.gz', cfg.PHYSIONET_JPG_CHEXPERT, 'https://physionet.org/files/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-chexpert.csv.gz'),
    ]
    missing = [(n, p, u) for n, p, u in needed if not os.path.exists(p)]
    if not missing:
        print("  [OK] All PhysioNet metadata present.")
        return
    print(f"  [DOWNLOAD] {len(missing)} files missing.")
    meta_dir = os.path.dirname(missing[0][1])
    os.makedirs(meta_dir, exist_ok=True)
    user, pwd = get_physionet_credentials()
    try:
        import requests
        r = requests.head('https://physionet.org/content/mimic-cxr-jpg/2.1.0/', auth=(user, pwd), timeout=30)
        if r.status_code == 403: raise RuntimeError("PhysioNet auth failed")
        print(f"  [AUTH] OK (HTTP {r.status_code})")
    except RuntimeError: raise
    except: pass
    for name, path, url in missing:
        out = os.path.join(meta_dir, name)
        if not os.path.exists(out):
            print(f"    Downloading {name}...")
            wget_download(url, out, user, pwd)
    # Verify zip
    import zipfile
    zip_path = os.path.join(meta_dir, 'mimic-cxr-reports.zip')
    if os.path.exists(zip_path):
        try:
            with zipfile.ZipFile(zip_path) as zf: zf.testzip()
            print(f"  [OK] Reports zip verified")
        except:
            print(f"  [WARN] Reports zip corrupt, re-downloading...")
            os.remove(zip_path)
            wget_download('https://physionet.org/files/mimic-cxr/2.0.0/mimic-cxr-reports.zip', zip_path, user, pwd)
    print("  [OK] All metadata downloaded.")

def load_physionet_metadata(cfg):
    print("  Loading metadata files...")
    records = pd.read_csv(cfg.PHYSIONET_RECORD_LIST)
    print(f"    {len(records):,} image records")
    studies = pd.read_csv(cfg.PHYSIONET_STUDY_LIST)
    print(f"    {len(studies):,} studies")
    jpg_meta = pd.read_csv(cfg.PHYSIONET_JPG_METADATA)
    print(f"    {len(jpg_meta):,} JPG metadata rows")
    jpg_split = pd.read_csv(cfg.PHYSIONET_JPG_SPLIT)
    print(f"    {len(jpg_split):,} split rows")
    chexpert = pd.read_csv(cfg.PHYSIONET_JPG_CHEXPERT)
    print(f"    {len(chexpert):,} CheXpert-labeled studies")
    required = ['study_id', 'Pneumothorax', 'Pneumonia', 'No Finding']
    missing_cols = [c for c in required if c not in chexpert.columns]
    if missing_cols: raise RuntimeError(f"CheXpert missing columns: {missing_cols}")
    df = records.merge(jpg_meta, on='dicom_id', how='inner')
    df = df.merge(jpg_split, on='dicom_id', how='inner')
    chexpert_no_subj = chexpert.drop(columns=['subject_id'], errors='ignore')
    df = df.merge(chexpert_no_subj, on='study_id', how='left')
    print(f"  Joined: {len(df):,} rows")
    return df, studies

def load_report_texts(studies, cfg):
    print(f"  Loading report texts from {cfg.PHYSIONET_REPORTS_ZIP}...")
    report_texts = {}
    with zipfile.ZipFile(cfg.PHYSIONET_REPORTS_ZIP) as z:
        names = z.namelist()
        print(f"    {len(names):,} files in zip")
        for _, row in tqdm(studies.iterrows(), total=len(studies), desc='Reports'):
            sid = row['study_id']; subj = row['subject_id']
            p = f"files/p{str(subj)[:2]}/p{subj}/s{sid}.txt"
            if p in names:
                try: report_texts[sid] = z.read(p).decode('utf-8', errors='replace')
                except: pass
    print(f"    Loaded {len(report_texts):,} report texts")
    return report_texts

def build_cohort(df, report_texts, cfg):
    print("  Building cohort from official report text...")
    all_records = []
    stats = {'total': 0, 'no_report': 0, 'no_impression': 0, 'other': 0, 'kept': 0}
    for study_id, study_group in tqdm(df.groupby('study_id'), desc='Cohort'):
        report = report_texts.get(study_id, None)
        stats['total'] += 1
        if not report: stats['no_report'] += 1; continue
        findings, impression = split_report_sections(report)
        if not impression or len(impression) < 10: stats['no_impression'] += 1; continue
        no_finding = study_group['No Finding'].iloc[0]
        pn_label = study_group['Pneumonia'].iloc[0]
        pnx = study_group['Pneumothorax'].iloc[0]
        if no_finding == 1.0: label = 0
        elif pn_label == 1.0: label = 1
        elif pnx == 1.0: label = 2
        else: label = -1
        if label == -1: stats['other'] += 1; continue
        for _, img_row in study_group.iterrows():
            if cfg.FRONTAL_ONLY and img_row.get('ViewPosition', 'AP') not in ('AP', 'PA'): continue
            all_records.append({
                'dicom_id': img_row['dicom_id'], 'subject_id': img_row['subject_id'],
                'study_id': study_id, 'findings': findings, 'impression': impression,
                'label': label, 'label_name': cfg.LABEL_NAMES[label],
                'view_position': img_row.get('ViewPosition', 'Unknown'),
                'split': img_row.get('split', 'unknown'), 'source': 'physionet',
            })
            stats['kept'] += 1
    print(f"  Cohort stats (before cap): {stats}")
    print(f"\n  PRE-CAP RAW CLASS COUNTS:")
    df_pre = pd.DataFrame(all_records)
    if len(df_pre) > 0:
        for label, name in enumerate(cfg.LABEL_NAMES):
            n = len(df_pre[df_pre['label'] == label])
            print(f"    {name}: {n:,}")
    caps = {0: cfg.NORMAL_CAP, 1: cfg.PNEUMONIA_CAP, 2: cfg.PNEUMOTHORAX_CAP}
    print(f"\n  Applying caps: {caps}")
    capped = []
    class_totals = defaultdict(int)
    for _, row in df_pre.iterrows():
        cap = caps.get(row['label'])
        if cap is not None and class_totals[row['label']] >= cap: continue
        class_totals[row['label']] += 1
        capped.append(row)
    df_capped = pd.DataFrame(capped)
    print(f"  After cap: {len(df_capped):,} (was {len(df_pre):,})")
    print(f"  Class distribution: {dict(df_capped['label_name'].value_counts())}")
    return df_capped

# ============================================================
# Execute: Download + Load + Cohort + Split + Images + Save
# ============================================================
print("=" * 70)
print("  PHASE 1: PhysioNet metadata + cohort construction")
print("=" * 70)
ensure_physionet_metadata(Config)
mimic_meta_df, studies_df = load_physionet_metadata(Config)
report_texts = load_report_texts(studies_df, Config)
mimic_df = build_cohort(mimic_meta_df, report_texts, Config)
print(f"\nTotal cohort: {len(mimic_df):,}")
print(f"  Unique patients: {mimic_df['subject_id'].nunique():,}")

# ============================================================
# Split: GroupShuffleSplit 70/15/15 + save final_split.pkl
# ============================================================
print("\n" + "=" * 70)
print("  SPLIT: GroupShuffleSplit 70/15/15 by patient")
print("=" * 70)
def report_fingerprint(text):
    return frozenset(re.findall(r'\b[a-z]+\b', text.lower()))
mimic_df['report_fingerprint'] = mimic_df['findings'].apply(report_fingerprint)
before = len(mimic_df)
deduped_parts = []
for split_name, split_df in mimic_df.groupby('split'):
    before_split = len(split_df)
    split_deduped = split_df.drop_duplicates(subset='report_fingerprint', keep='first')
    print(f"  Dedup {split_name}: {before_split:,} -> {len(split_deduped):,}")
    deduped_parts.append(split_deduped)
mimic_df = pd.concat(deduped_parts, ignore_index=True)
print(f"  Total dedup: {before:,} -> {len(mimic_df):,}")
# Cross-split fingerprint overlap check
print(f"\n  Cross-split fingerprint overlap check...")
train_fps = set(mimic_df[mimic_df['split'] == 'train']['report_fingerprint'])
val_fps = set(mimic_df[mimic_df['split'] == 'validation']['report_fingerprint'])
test_fps = set(mimic_df[mimic_df['split'] == 'test']['report_fingerprint'])
if len(val_fps) == 0:
    val_fps = set(mimic_df[mimic_df['split'] == 'validate']['report_fingerprint'])
overlap = len(train_fps & val_fps) + len(train_fps & test_fps) + len(val_fps & test_fps)
if overlap > 0:
    print(f"  [WARN] {overlap} cross-split duplicates. Removing from val/test...")
    keep_mask = ~((mimic_df['split'].isin(['validation', 'validate', 'test'])) & (mimic_df['report_fingerprint'].isin(train_fps)))
    mimic_df = mimic_df[keep_mask].reset_index(drop=True)
    print(f"  [OK] Removed {overlap} cross-split duplicates")
else:
    print(f"  [OK] Zero cross-split fingerprint overlap")
# GroupShuffleSplit 70/15/15
print(f"\n  Building 70/15/15 patient-level split...")
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss1.split(mimic_df, groups=mimic_df['subject_id']))
temp_df = mimic_df.iloc[temp_idx].reset_index(drop=True)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
rel_val_idx, rel_test_idx = next(gss2.split(temp_df, groups=temp_df['subject_id']))
train_df = mimic_df.iloc[train_idx].reset_index(drop=True)
val_df = temp_df.iloc[rel_val_idx].reset_index(drop=True)
test_df = temp_df.iloc[rel_test_idx].reset_index(drop=True)
train_df['split'] = 'train'
val_df['split'] = 'validation'
test_df['split'] = 'test'
print(f"\n  FINAL Split: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")
for name, sub in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f"    {name}: {dict(sub['label_name'].value_counts())}")
train_s = set(train_df['subject_id'].unique())
val_s = set(val_df['subject_id'].unique())
test_s = set(test_df['subject_id'].unique())
print(f"\n  Patient overlap: TV={len(train_s & val_s)}, TT={len(train_s & test_s)}, VT={len(val_s & test_s)}")
assert len(train_s & val_s) == 0 and len(train_s & test_s) == 0 and len(val_s & test_s) == 0
print(f"  [OK] Zero patient overlap confirmed.")

# ============================================================
# Image download from authenticated PhysioNet
# ============================================================
print("\n" + "=" * 70)
print("  IMAGE DOWNLOAD (authenticated PhysioNet)")
print("=" * 70)
img_dir = Config.PHYSIONET_IMAGE_DIR
os.makedirs(img_dir, exist_ok=True)
n_existing = len([f for f in os.listdir(img_dir) if f.endswith('.jpg')]) if os.path.exists(img_dir) else 0
print(f"  {n_existing} images already on disk")
if n_existing < len(mimic_df) * 0.5:
    user, pwd = get_physionet_credentials()
    sample = mimic_df.iloc[0]
    test_url = f"{Config.PHYSIONET_IMAGE_BASE}files/p{str(sample['subject_id'])[:2]}/p{sample['subject_id']}/s{sample['study_id']}/{sample['dicom_id']}.jpg"
    test_out = os.path.join(img_dir, f"test_{sample['dicom_id']}.jpg")
    print(f"  [TEST] Testing auth on single image...")
    test_cmd = ['wget', '-q', '--user=' + user, '--password=' + pwd, '-O', test_out, test_url]
    test_result = subprocess.run(test_cmd, capture_output=True, text=True, timeout=30)
    if test_result.returncode == 0 and os.path.exists(test_out) and os.path.getsize(test_out) > 1000:
        print(f"  [TEST] OK ({os.path.getsize(test_out)/1e3:.0f} KB)")
        os.remove(test_out)
    else:
        raise RuntimeError(f"PhysioNet auth failed: {test_result.stderr[:200]}")
    to_download = []
    for _, row in mimic_df.iterrows():
        out_path = os.path.join(img_dir, f"{row['dicom_id']}.jpg")
        if not os.path.exists(out_path) or os.path.getsize(out_path) < 1000:
            url = f"{Config.PHYSIONET_IMAGE_BASE}files/p{str(row['subject_id'])[:2]}/p{row['subject_id']}/s{row['study_id']}/{row['dicom_id']}.jpg"
            to_download.append((url, out_path))
    print(f"  {len(to_download)} images to download (rest cached)")
    if to_download:
        import concurrent.futures
        def download_one(args):
            url, out_path = args
            if os.path.exists(out_path) and os.path.getsize(out_path) > 1000: return True
            cmd = ['wget', '-q', '-c', '--user=' + user, '--password=' + pwd, '--timeout=30', '--tries=3', '-O', out_path, url]
            try:
                r = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
                ok = r.returncode == 0 and os.path.exists(out_path) and os.path.getsize(out_path) > 1000
                if not ok and os.path.exists(out_path): os.remove(out_path)
                return ok
            except:
                if os.path.exists(out_path): os.remove(out_path)
                return False
        print(f"  Downloading with 3 parallel workers...")
        downloaded, failed = 0, 0
        with concurrent.futures.ThreadPoolExecutor(max_workers=3) as ex:
            futures = {ex.submit(download_one, a): a for a in to_download}
            for i, f in enumerate(concurrent.futures.as_completed(futures), 1):
                if f.result(): downloaded += 1
                else: failed += 1
                if i % 50 == 0: print(f"    {i}/{len(to_download)} ({downloaded} ok, {failed} failed)", flush=True)
        print(f"  Downloaded: {downloaded}, Failed: {failed}")
mimic_df['img_path'] = mimic_df.apply(lambda r: os.path.join(img_dir, f"{r['dicom_id']}.jpg"), axis=1)
for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    df['img_path'] = df.apply(lambda r: os.path.join(img_dir, f"{r['dicom_id']}.jpg"), axis=1)
    has_img = df['img_path'].apply(os.path.exists).sum()
    print(f"  {name}: {has_img}/{len(df)} images accessible")

# ============================================================
# Save all artifacts for Part 2 and Part 3
# ============================================================
print("\n" + "=" * 70)
print("  FINAL SAVE (Part 1)")
print("=" * 70)
with open('/kaggle/working/mimic_df.pkl', 'wb') as f:
    pickle.dump(mimic_df, f)
print(f"  [SAVED] mimic_df.pkl ({len(mimic_df):,} rows)")
phase0_state = {
    'TB_DECISION': TB_DECISION, 'PHASE0_FREEZE_LMH': PHASE0_FREEZE_LMH,
    'LABEL_NAMES': Config.LABEL_NAMES, 'SEED_LIST': Config.SEED_LIST,
    'ABLATION_SEED_LIST': Config.ABLATION_SEED_LIST,
}
with open('/kaggle/working/phase0_state.pkl', 'wb') as f:
    pickle.dump(phase0_state, f)
print(f"  [SAVED] phase0_state.pkl")
split_mapping = {
    'train': train_df['dicom_id'].tolist(),
    'validation': val_df['dicom_id'].tolist(),
    'test': test_df['dicom_id'].tolist(),
}
with open('/kaggle/working/final_split.pkl', 'wb') as f:
    pickle.dump(split_mapping, f)
print(f"  [SAVED] final_split.pkl")
print(f"\n{'='*70}")
print(f"  PART 1 COMPLETE")
print(f"{'='*70}")
print(f"\n  NEXT STEPS:")
print(f"  1. Save Version > Quick Save")
print(f"  2. Attach this output to Part 2")
print(f"  3. Run Part 2 (training)")


Report splitter defined.
  PHASE 1: PhysioNet metadata + cohort construction
  [DOWNLOAD] 6 files missing.
  [OK] PhysioNet credentials loaded
  [AUTH] OK (HTTP 200)
    [OK] cxr-record-list.csv.gz (14.9 MB)
    [OK] cxr-study-list.csv.gz (2.3 MB)
    [OK] mimic-cxr-reports.zip (141.9 MB)
    [OK] mimic-cxr-2.0.0-metadata.csv.gz (16.5 MB)
    [OK] mimic-cxr-2.0.0-split.csv.gz (12.2 MB)
    [OK] mimic-cxr-2.0.0-chexpert.csv.gz (2.1 MB)
  [OK] Reports zip verified
  [OK] All metadata downloaded.
  Loading metadata files...
    377,110 image records
    227,835 studies
    377,110 JPG metadata rows
    377,110 split rows
    227,827 CheXpert-labeled studies
  Joined: 377,110 rows
  Loading report texts from /kaggle/working/physionet_metadata/mimic-cxr-reports.zip...
    293,234 files in zip


Reports:   0%|          | 0/227835 [00:00<?, ?it/s]

    Loaded 227,835 report texts
  Building cohort from official report text...


Cohort:   0%|          | 0/227835 [00:00<?, ?it/s]

  Cohort stats (before cap): {'total': 227835, 'no_report': 0, 'no_impression': 629, 'other': 125716, 'kept': 108626}

  PRE-CAP RAW CLASS COUNTS:
    Normal: 80,472
    Pneumonia: 17,221
    Pneumothorax: 10,933

  Applying caps: {0: 2500, 1: 2500, 2: 1000}
  After cap: 6,000 (was 108,626)
  Class distribution: {'Normal': np.int64(2500), 'Pneumonia': np.int64(2500), 'Pneumothorax': np.int64(1000)}

Total cohort: 6,000
  Unique patients: 4,693

  SPLIT: GroupShuffleSplit 70/15/15 by patient
  Dedup test: 94 -> 55
  Dedup train: 5,861 -> 2,910
  Dedup validate: 45 -> 28
  Total dedup: 6,000 -> 2,993

  Cross-split fingerprint overlap check...
  [WARN] 36 cross-split duplicates. Removing from val/test...
  [OK] Removed 36 cross-split duplicates

  Building 70/15/15 patient-level split...

  FINAL Split: Train=2055, Val=450, Test=457
    Train: {'Normal': np.int64(1112), 'Pneumonia': np.int64(738), 'Pneumothorax': np.int64(205)}
    Val: {'Normal': np.int64(245), 'Pneumonia': np.int64(172